In [0]:
# Create gold, silver and bronze, run only once


CATALOG = "final_project_databi_group8"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{CATALOG}`.`bronze`")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{CATALOG}`.`silver`")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{CATALOG}`.`gold`")

print(f"✓ {CATALOG}.bronze ready")
print(f"✓ {CATALOG}.silver ready")
print(f"✓ {CATALOG}.gold ready")

In [0]:
# query to check data in bronze 


CATALOG = "final_project_databi_group8"

#  Chicago Bronze 
print("=== CHICAGO BRONZE ===")
display(spark.sql(f"""
    SELECT
        city_source,
        LEFT(_batch_id, 8)          AS batch_short,
        _extract_date,
        _run_id,
        _schema_version,
        COUNT(*)                    AS row_count
    FROM {CATALOG}.bronze.chicago_raw
    GROUP BY city_source, _batch_id, _extract_date, _run_id, _schema_version
    ORDER BY _extract_date DESC
"""))

#  Dallas Bronze
print("=== DALLAS BRONZE ===")
display(spark.sql(f"""
    SELECT
        city_source,
        LEFT(_batch_id, 8)          AS batch_short,
        _extract_date,
        _run_id,
        _schema_version,
        COUNT(*)                    AS row_count
    FROM {CATALOG}.bronze.dallas_raw
    GROUP BY city_source, _batch_id, _extract_date, _run_id, _schema_version
    ORDER BY _extract_date DESC
"""))

#  Quick sanity check
chi_count = spark.table(f"{CATALOG}.bronze.chicago_raw").count()
dal_count = spark.table(f"{CATALOG}.bronze.dallas_raw").count()

print(f"\n Chicago Bronze : {chi_count:,} rows ")
print(f"\n Dallas Bronze  : {dal_count:,} rows ")

In [0]:
CATALOG = "final_project_databi_group8"

# ── Chicago Silver Clean ─────────────────────────────────────
print("=== CHICAGO SILVER CLEAN ===")
display(spark.sql(f"""
    SELECT
        city_source,
        COUNT(*)                                            AS total_inspections,
        ROUND(AVG(inspection_score), 2)                     AS avg_score,
        ROUND(AVG(violation_count), 2)                      AS avg_violations,
        SUM(CASE WHEN inspection_result = 'Pass'               THEN 1 ELSE 0 END) AS pass_count,
        SUM(CASE WHEN inspection_result = 'Fail'               THEN 1 ELSE 0 END) AS fail_count,
        SUM(CASE WHEN inspection_result = 'Pass w/ Conditions' THEN 1 ELSE 0 END) AS pass_cond,
        SUM(CASE WHEN risk_category = 'High'    THEN 1 ELSE 0 END) AS high_risk,
        SUM(CASE WHEN risk_category = 'Medium'  THEN 1 ELSE 0 END) AS medium_risk,
        SUM(CASE WHEN risk_category = 'Low'     THEN 1 ELSE 0 END) AS low_risk,
        MIN(inspection_date)                                AS earliest_date,
        MAX(inspection_date)                                AS latest_date
    FROM {CATALOG}.silver.chicago_clean
    GROUP BY city_source
"""))

# ── Chicago Quarantine ───────────────────────────────────────
print("=== CHICAGO QUARANTINE ===")
display(spark.sql(f"""
    SELECT
        _dqx_failed_checks,
        COUNT(*) AS row_count
    FROM {CATALOG}.silver.chicago_quarantine
    GROUP BY _dqx_failed_checks
    ORDER BY row_count DESC
    LIMIT 10
"""))

# ── Dallas Silver Clean ──────────────────────────────────────
print("=== DALLAS SILVER CLEAN ===")
display(spark.sql(f"""
    SELECT
        city_source,
        COUNT(*)                                            AS total_inspections,
        ROUND(AVG(inspection_score), 2)                     AS avg_score,
        ROUND(AVG(violation_count), 2)                      AS avg_violations,
        MIN(inspection_score)                               AS min_score,
        MAX(inspection_score)                               AS max_score,
        SUM(CASE WHEN inspection_type = 'Routine'   THEN 1 ELSE 0 END) AS routine,
        SUM(CASE WHEN inspection_type = 'Follow-up' THEN 1 ELSE 0 END) AS followup,
        SUM(CASE WHEN inspection_type = 'Complaint' THEN 1 ELSE 0 END) AS complaint,
        MIN(inspection_date)                                AS earliest_date,
        MAX(inspection_date)                                AS latest_date
    FROM {CATALOG}.silver.dallas_clean
    GROUP BY city_source
"""))

# ── Dallas Quarantine ────────────────────────────────────────
print("=== DALLAS QUARANTINE ===")
display(spark.sql(f"""
    SELECT
        _dqx_failed_checks,
        COUNT(*) AS row_count
    FROM {CATALOG}.silver.dallas_quarantine
    GROUP BY _dqx_failed_checks
    ORDER BY row_count DESC
    LIMIT 10
"""))

# ── Violations ───────────────────────────────────────────────
print("=== VIOLATIONS ===")
display(spark.sql(f"""
    SELECT
        city_source,
        COUNT(*)                        AS total_violation_rows,
        COUNT(DISTINCT violation_code)  AS unique_codes,
        COUNT(DISTINCT inspection_id)   AS inspections_with_violations
    FROM {CATALOG}.silver.chicago_violations
    GROUP BY city_source
"""))

display(spark.sql(f"""
    SELECT
        city_source,
        COUNT(*)                        AS total_violation_rows,
        COUNT(DISTINCT violation_code)  AS unique_codes,
        COUNT(DISTINCT inspection_id)   AS inspections_with_violations
    FROM {CATALOG}.silver.dallas_violations
    GROUP BY city_source
"""))

# ── Row count summary ────────────────────────────────────────
chi_clean  = spark.table(f"{CATALOG}.silver.chicago_clean").count()
dal_clean  = spark.table(f"{CATALOG}.silver.dallas_clean").count()
chi_quar   = spark.table(f"{CATALOG}.silver.chicago_quarantine").count()
dal_quar   = spark.table(f"{CATALOG}.silver.dallas_quarantine").count()
chi_viols  = spark.table(f"{CATALOG}.silver.chicago_violations").count()
dal_viols  = spark.table(f"{CATALOG}.silver.dallas_violations").count()

print(f"\n{'='*50}")
print(f"SILVER SUMMARY")
print(f"{'='*50}")
print(f"Chicago clean      : {chi_clean:,}  (expected ~253,000)")
print(f"Chicago quarantine : {chi_quar:,}   (expected ~86,000+)")
print(f"Chicago violations : {chi_viols:,}")
print(f"Dallas clean       : {dal_clean:,}   (expected ~54,000)")
print(f"Dallas quarantine  : {dal_quar:,}    (expected ~25,000)")
print(f"Dallas violations  : {dal_viols:,}")
print(f"{'='*50}")
print(f"Total clean rows   : {chi_clean + dal_clean:,}")

In [0]:
# cell to check if everything is implemented as per architechture 

CATALOG = "final_project_databi_group8"
results = []

def check(name, passed, detail=""):
    status = "PASS" if passed else "FAIL"
    results.append((name, status, detail))
    icon = "✓" if passed else "✗"
    print(f"  {icon} [{status}] {name} {detail}")

print("=" * 60)
print("BRONZE LAYER CHECKS")
print("=" * 60)

# Row counts
chi_bronze = spark.table(f"{CATALOG}.bronze.chicago_raw").count()
dal_bronze = spark.table(f"{CATALOG}.bronze.dallas_raw").count()
check("Chicago Bronze row count", chi_bronze == 308357, f"({chi_bronze:,})")
check("Dallas Bronze row count",  dal_bronze == 78984,  f"({dal_bronze:,})")

# Audit columns exist
chi_cols = spark.table(f"{CATALOG}.bronze.chicago_raw").columns
dal_cols = spark.table(f"{CATALOG}.bronze.dallas_raw").columns
for col in ["_batch_id","_extract_ts","_extract_date","_source_file",
            "_run_id","_schema_version","_row_hash","inspection_id","city_source"]:
    check(f"Bronze CHI has {col}", col in chi_cols)
    check(f"Bronze DAL has {col}", col in dal_cols)

# _extract_ts is TIMESTAMP not STRING
from pyspark.sql.types import TimestampType
chi_ts_type = [f.dataType for f in spark.table(f"{CATALOG}.bronze.chicago_raw").schema if f.name == "_extract_ts"]
check("_extract_ts is TIMESTAMP", chi_ts_type and isinstance(chi_ts_type[0], TimestampType),
      f"(type: {chi_ts_type[0] if chi_ts_type else 'NOT FOUND'})")

print()
print("=" * 60)
print("SILVER LAYER CHECKS")
print("=" * 60)

chi_clean = spark.table(f"{CATALOG}.silver.chicago_clean")
dal_clean = spark.table(f"{CATALOG}.silver.dallas_clean")
chi_quar  = spark.table(f"{CATALOG}.silver.chicago_quarantine")
dal_quar  = spark.table(f"{CATALOG}.silver.dallas_quarantine")
chi_viols = spark.table(f"{CATALOG}.silver.chicago_violations")
dal_viols = spark.table(f"{CATALOG}.silver.dallas_violations")

# Row counts reasonable
chi_clean_count = chi_clean.count()
dal_clean_count = dal_clean.count()
check("Chicago Silver has rows",  chi_clean_count > 200000, f"({chi_clean_count:,})")
check("Dallas Silver has rows",   dal_clean_count > 40000,  f"({dal_clean_count:,})")
check("Chicago quarantine exists",chi_quar.count() > 0)
check("Dallas quarantine exists", dal_quar.count() > 0)

# Required Silver columns
silver_required = ["inspection_id","restaurant_name","inspection_date",
                   "inspection_type","inspection_score","zip_code",
                   "city_source","violation_count","_batch_id",
                   "_silver_ts","_silver_date","_dqx_status",
                   "_silver_version","_bronze_extract_ts"]
chi_silver_cols = chi_clean.columns
dal_silver_cols = dal_clean.columns
for col in silver_required:
    check(f"Silver CHI has {col}", col in chi_silver_cols)
    check(f"Silver DAL has {col}", col in dal_silver_cols)

# violations_raw should NOT be in clean table
check("violations_raw NOT in chicago_clean", "violations_raw" not in chi_silver_cols)

# _dqx_failed_checks in quarantine
chi_quar_cols = chi_quar.columns
dal_quar_cols = dal_quar.columns
check("_dqx_failed_checks in CHI quarantine", "_dqx_failed_checks" in chi_quar_cols)
check("_dqx_failed_checks in DAL quarantine", "_dqx_failed_checks" in dal_quar_cols)

# No nulls on critical columns
from pyspark.sql import functions as F
chi_null_name = chi_clean.filter(F.col("restaurant_name").isNull()).count()
chi_null_date = chi_clean.filter(F.col("inspection_date").isNull()).count()
dal_null_name = dal_clean.filter(F.col("restaurant_name").isNull()).count()
dal_null_date = dal_clean.filter(F.col("inspection_date").isNull()).count()
check("No null restaurant_name in CHI clean", chi_null_name == 0, f"({chi_null_name} nulls)")
check("No null inspection_date in CHI clean", chi_null_date == 0, f"({chi_null_date} nulls)")
check("No null restaurant_name in DAL clean", dal_null_name == 0, f"({dal_null_name} nulls)")
check("No null inspection_date in DAL clean", dal_null_date == 0, f"({dal_null_date} nulls)")

# Score range for Dallas
dal_bad_score = dal_clean.filter(
    (F.col("inspection_score") < 0) | (F.col("inspection_score") > 100)
).count()
check("Dallas scores all 0-100", dal_bad_score == 0, f"({dal_bad_score} out of range)")

# Score>=90 with >3 violations quarantined
dal_viol_rule = dal_clean.filter(
    (F.col("inspection_score") >= 90) & (F.col("violation_count") > 3)
).count()
check("No score>=90 with >3 violations in DAL clean", dal_viol_rule == 0,
      f"({dal_viol_rule} violations of rule)")

# Violations tables have rows
check("Chicago violations table has rows", chi_viols.count() > 0)
check("Dallas violations table has rows",  dal_viols.count() > 0)

print()
print("=" * 60)
print("UNIFIED SILVER CHECKS")
print("=" * 60)

unified = spark.table(f"{CATALOG}.silver.inspections_unified")
unified_viols = spark.table(f"{CATALOG}.silver.violations_unified")

unified_count = unified.count()
check("inspections_unified has rows", unified_count > 0, f"({unified_count:,})")
check("violations_unified has rows",  unified_viols.count() > 0)

# Both cities present
cities = [r[0] for r in unified.select("city_source").distinct().collect()]
check("CHI present in unified", "CHI" in cities)
check("DAL present in unified", "DAL" in cities)

# _merged_ts exists
check("_merged_ts in unified", "_merged_ts" in unified.columns)

print()
print("=" * 60)
print("AUDIT SUMMARY")
print("=" * 60)
passed = sum(1 for _, s, _ in results if s == "PASS")
failed = sum(1 for _, s, _ in results if s == "FAIL")
print(f"  PASSED : {passed}")
print(f"  FAILED : {failed}")
print(f"  TOTAL  : {len(results)}")
if failed == 0:
    print("\n  ALL CHECKS PASSED — pipeline meets industry standards")
else:
    print(f"\n  {failed} checks need attention:")
    for name, status, detail in results:
        if status == "FAIL":
            print(f"    ✗ {name} {detail}")

In [0]:
CATALOG = "final_project_databi_group8"

# Check for duplicate inspection_ids in unified Silver
dup_check = spark.sql(f"""
    SELECT
        city_source,
        COUNT(*)                        AS total_rows,
        COUNT(DISTINCT inspection_id)   AS unique_inspection_ids,
        COUNT(*) - COUNT(DISTINCT inspection_id) AS duplicates
    FROM {CATALOG}.silver.inspections_unified
    GROUP BY city_source
    ORDER BY city_source
""")
display(dup_check)

In [0]:
CATALOG = "final_project_databi_group8"

# Check if dim_location has duplicate business keys
display(spark.sql(f"""
    SELECT
        street_address, zip_code, city_source,
        COUNT(*) AS cnt
    FROM {CATALOG}.gold.dim_location
    GROUP BY street_address, zip_code, city_source
    HAVING COUNT(*) > 1
    ORDER BY cnt DESC
    LIMIT 10
"""))

In [0]:
# Check if dim_restaurant has duplicate business keys for is_current=true
display(spark.sql(f"""
    SELECT
        restaurant_name, street_address, city_source,
        COUNT(*) AS cnt
    FROM {CATALOG}.gold.dim_restaurant
    WHERE is_current = true
    GROUP BY restaurant_name, street_address, city_source
    HAVING COUNT(*) > 1
    ORDER BY cnt DESC
    LIMIT 10
"""))

In [0]:
CATALOG = "final_project_databi_group8"
spark.sql(f"DROP TABLE IF EXISTS {CATALOG}.gold.dim_location")
spark.sql(f"DROP TABLE IF EXISTS {CATALOG}.gold.fact_inspection")
spark.sql(f"DROP TABLE IF EXISTS {CATALOG}.gold.fact_violation")
print("Tables dropped — re-run 06_gold_dimensions then 07_gold_facts")

In [0]:
CATALOG = "final_project_databi_group8"
spark.sql(f"DROP TABLE IF EXISTS {CATALOG}.gold.dim_violation")
spark.sql(f"DROP TABLE IF EXISTS {CATALOG}.gold.fact_violation")
print("Dropped — ready to rebuild")

In [0]:
display(spark.sql(f"""
    SELECT
        dr.restaurant_name,
        dr.facility_type,
        dl.city,
        dl.zip_code,
        dd.year,
        dd.month_name,
        fi.inspection_result,
        fi.inspection_score,
        fi.violation_count,
        fi.city_source
    FROM {CATALOG}.gold.fact_inspection fi
    JOIN {CATALOG}.gold.dim_restaurant dr
        ON fi.restaurant_key = dr.restaurant_key
        AND dr.is_current = true
    JOIN {CATALOG}.gold.dim_location dl
        ON fi.location_key = dl.location_key
    JOIN {CATALOG}.gold.dim_date dd
        ON fi.date_key = dd.date_key
    LIMIT 20
"""))